# Selah — technical evidence notebook

**Competition:** Scripture in New Frontiers  
**Artifact scope:** an offline, standard-library contract test for the Selah prototype.

This notebook uses only fictional fixtures. It has no network code, credentials, personal data, external datasets, models, wheels, or provider output. It does **not** claim that Gloo or YouVersion was called from this notebook. Its purpose is narrower: make the safety boundary and deterministic validation logic inspectable and reproducible with Internet disabled.

## One voluntary pause, with agency intact

The demo scenario is deliberately synthetic: a creator is about to send a heated reply and opts into an eight-second pause. Selah does not block, rewrite, or post. The untouched draft remains available, and the person chooses whether to edit it.

### Intended live architecture

1. A fixed fictional draft and one user intent reach a local same-origin backend.
2. Gloo AI is asked to return exactly two opaque enum values: one `passage_key` and one `question_key`. It is never asked to quote or paraphrase Scripture.
3. Strict JSON parsing rejects duplicate keys, non-finite values, extra fields, unknown enums, tool calls, or incomplete responses.
4. The validated passage key maps server-side to one fixed USFM reference.
5. YouVersion is the sole source of displayed passage text, reference, version, link, and copyright. Returned identifiers and attribution must match the request.
6. On any provider or validation failure, Selah fails open: no verse is shown, the original draft is untouched, and ordinary posting remains available.

Provider connectivity and competition entitlement are separate release gates. The code path remains disabled until zero-cost access is confirmed without a payment method.

In [ ]:
import json
from urllib.parse import urlsplit


class ContractError(ValueError):
    pass


PASSAGE_KEYS = frozenset({
    "listen_first", "gentle_answer", "build_up", "make_peace",
    "peacemaker", "careful_words", "few_words", "honor_others",
    "do_not_repay", "gracious_words", "guard_my_words",
    "bear_and_forgive",
})
QUESTION_KEYS = frozenset({
    "hear_meaning", "protect_bond", "separate_fact",
    "lower_heat", "invite_dialogue", "own_part",
})


def strict_json_loads(raw):
    def unique_object(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ContractError("duplicate key")
            result[key] = value
        return result

    def reject_nonfinite(_):
        raise ContractError("non-finite number")

    try:
        return json.loads(
            raw, object_pairs_hook=unique_object, parse_constant=reject_nonfinite
        )
    except (json.JSONDecodeError, TypeError) as exc:
        raise ContractError("invalid JSON") from exc


def validate_gloo_choice(raw):
    value = strict_json_loads(raw)
    if not isinstance(value, dict) or set(value) != {"passage_key", "question_key"}:
        raise ContractError("schema mismatch")
    passage_key = value["passage_key"]
    question_key = value["question_key"]
    if not isinstance(passage_key, str) or passage_key not in PASSAGE_KEYS:
        raise ContractError("passage outside allowlist")
    if not isinstance(question_key, str) or question_key not in QUESTION_KEYS:
        raise ContractError("question outside allowlist")
    return passage_key, question_key

In [ ]:
gloo_fixtures = [
    ('{"passage_key":"gentle_answer","question_key":"lower_heat"}', True),
    ('{"passage_key":"gentle_answer","passage_key":"listen_first","question_key":"lower_heat"}', False),
    ('{"passage_key":"gentle_answer","question_key":"lower_heat","verse":"untrusted"}', False),
    ('{"passage_key":"gentle_answer","question_key":"lower_heat","score":NaN}', False),
    ('{"passage_key":"invented_reference","question_key":"lower_heat"}', False),
    ('{"passage_key":"gentle_answer","question_key":"invented_question"}', False),
    ('{"passage_key":42,"question_key":"lower_heat"}', False),
]

passed = 0
for raw, should_accept in gloo_fixtures:
    try:
        validate_gloo_choice(raw)
        accepted = True
    except ContractError:
        accepted = False
    assert accepted is should_accept
    passed += 1

print(f"Gloo key-only contract fixtures: {passed}/{len(gloo_fixtures)} passed")

## Exact-text and attribution contract

The next fixture contains a sentinel string, **not Bible text**. The validator demonstrates the intended invariant: content is carried through byte-for-byte from the validated YouVersion response; it is never generated, corrected, or completed locally. The passage ID, Bible ID, attribution fields, and canonical `www.bible.com` HTTPS link must all be valid before display.

In [ ]:
def validate_youversion_contract(metadata_raw, passage_raw, expected_bible_id, expected_passage_id):
    metadata = strict_json_loads(metadata_raw)
    passage = strict_json_loads(passage_raw)
    if not isinstance(metadata, dict) or metadata.get("id") != expected_bible_id:
        raise ContractError("Bible ID mismatch")
    required = ["title", "abbreviation", "copyright", "youversion_deep_link"]
    if any(not isinstance(metadata.get(key), str) or not metadata[key] for key in required):
        raise ContractError("incomplete attribution")
    link = urlsplit(metadata["youversion_deep_link"])
    if (
        link.scheme != "https"
        or link.hostname != "www.bible.com"
        or link.username is not None
        or link.password is not None
        or link.port not in (None, 443)
    ):
        raise ContractError("non-canonical link")
    if not isinstance(passage, dict) or passage.get("id") != expected_passage_id:
        raise ContractError("passage ID mismatch")
    content = passage.get("content")
    reference = passage.get("reference")
    if not isinstance(content, str) or not content or len(content) > 20_000:
        raise ContractError("invalid exact text")
    if not isinstance(reference, str) or not reference or len(reference) > 200:
        raise ContractError("invalid reference")
    return {
        "content": content,
        "reference": reference,
        "version": metadata["abbreviation"],
        "version_title": metadata["title"],
        "copyright": metadata["copyright"],
        "youversion_url": metadata["youversion_deep_link"],
    }


metadata_fixture = json.dumps({
    "id": 900001,
    "title": "Synthetic Bible Metadata Fixture",
    "abbreviation": "SYN",
    "copyright": "Synthetic attribution for contract testing only",
    "youversion_deep_link": "https://www.bible.com/versions/900001",
})
sentinel = "SYNTHETIC_PROVIDER_TEXT_NOT_SCRIPTURE"
passage_fixture = json.dumps({
    "id": "PRO.15.1",
    "content": sentinel,
    "reference": "Synthetic reference",
})

valid = validate_youversion_contract(metadata_fixture, passage_fixture, 900001, "PRO.15.1")
assert valid["content"] == sentinel

rejections = 0
invalid_cases = [
    (metadata_fixture, passage_fixture, 900001, "JAS.1.19"),
    (metadata_fixture.replace("Synthetic attribution for contract testing only", ""), passage_fixture, 900001, "PRO.15.1"),
    (metadata_fixture.replace("www.bible.com", "www.bible.com.evil.invalid"), passage_fixture, 900001, "PRO.15.1"),
]
for args in invalid_cases:
    try:
        validate_youversion_contract(*args)
    except ContractError:
        rejections += 1
assert rejections == len(invalid_cases)
print(f"YouVersion exact-text fixtures: {1 + rejections}/{1 + len(invalid_cases)} passed")

## Reliability result and its limits

The pre-publication standard-library suite result is **33/33 tests passed**:

| Area | Tests | Evidence boundary |
|---|---:|---|
| Strict JSON | 4 | Valid JSON accepted; duplicate keys, NaN, and trailing text rejected |
| Request and privacy validation | 7 | Exact input keys/intents; email, phone, URL, and oversize input rejected |
| Gloo key-only output | 4 | Exact schema and passage/question allowlists |
| Offline behavior | 2 | All intents have a question; no verse is shown or invented |
| Live-adapter fixtures | 7 | Synthetic transport, in-memory token reuse, fixed draft, request budget, host/key boundaries |
| Static frontend safety | 4 | No local/session storage, `innerHTML`, inline script, or external HTML resource |
| Local HTTP integration | 5 | Security headers, no-verse flow, fail-open error, origin and session checks |
| **Total** | **33** | **Synthetic/local evidence only** |

The suite does not prove live API availability, provider uptime, user outcomes, or reduced harm. No such claim is made. In the application, every provider or validation exception becomes one generic fail-open response: the pause ends, no passage is displayed, and the original draft remains available. Drafts and credentials are not intentionally persisted by the prototype; provider handling remains governed by provider terms.

## Public artifacts

- Code: `PUBLIC_GITHUB_URL`
- Demo video: `PUBLIC_YOUTUBE_URL`
- Kaggle writeup: `PUBLIC_KAGGLE_WRITEUP_URL`

These placeholders must be replaced only after the corresponding sanitized public artifacts exist.